In [1]:
print("goodbye")

goodbye


In [2]:
# ============================================================
# Customer Return Prediction - Data Preparation
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ============================================================
# LOAD DATA (FAST TRAINING SAMPLE)
# ============================================================

print("Loading dataset...")

df = pd.read_csv("../data/clean_data.csv")

df["invoicedate"] = pd.to_datetime(df["invoicedate"])

# ------------------------------------------------------------
# SAMPLE CUSTOMERS (Keeps complete purchase history)
# ------------------------------------------------------------

np.random.seed(42)

sample_customers = np.random.choice(
    df["customer_id"].unique(),
    size=500,          # Change to 1000 if needed
    replace=False
)

df = df[df["customer_id"].isin(sample_customers)]

print(f"Dataset Shape : {df.shape}")
print(f"Customers     : {df['customer_id'].nunique()}")

# ============================================================
# CREATE TRAINING DATASET
# ============================================================

print("\nCreating Training Dataset...")

cutoff_date = df["invoicedate"].max() - pd.Timedelta(days=90)

history = df[df["invoicedate"] <= cutoff_date].copy()

future = df[df["invoicedate"] > cutoff_date].copy()

# ============================================================
# FEATURE ENGINEERING
# ============================================================

customer = (
    history.groupby("customer_id")
    .agg(
        TotalOrders=("invoice", "nunique"),
        TotalRevenue=("total_price", "sum"),
        TotalQuantity=("quantity", "sum"),
        FirstPurchase=("invoicedate", "min"),
        LastPurchase=("invoicedate", "max"),
    )
    .reset_index()
)

# ------------------------------------------------------------
# Average Order Value
# ------------------------------------------------------------

customer["AverageOrderValue"] = (
    customer["TotalRevenue"] /
    customer["TotalOrders"]
)

# ------------------------------------------------------------
# Customer Lifetime
# ------------------------------------------------------------

customer["CustomerLifetime"] = (
    customer["LastPurchase"] -
    customer["FirstPurchase"]
).dt.days

customer["CustomerLifetime"] = (
    customer["CustomerLifetime"]
    .replace(0, 1)
)

# ------------------------------------------------------------
# Recency
# ------------------------------------------------------------

customer["Recency"] = (
    cutoff_date -
    customer["LastPurchase"]
).dt.days

# ------------------------------------------------------------
# Purchase Frequency
# ------------------------------------------------------------

customer["PurchaseFrequency"] = (
    customer["TotalOrders"] /
    customer["CustomerLifetime"]
)

# ------------------------------------------------------------
# Country
# ------------------------------------------------------------

country = (
    history.groupby("customer_id")["country"]
    .first()
    .reset_index()
)

customer = customer.merge(
    country,
    on="customer_id",
    how="left"
)

# ============================================================
# TARGET VARIABLE
# ============================================================

print("Creating Target Variable...")

future_customers = future["customer_id"].unique()

customer["Target"] = (
    customer["customer_id"]
    .isin(future_customers)
    .astype(int)
)

print("\nTarget Distribution")

print(customer["Target"].value_counts())

# ============================================================
# ENCODING
# ============================================================

customer = pd.get_dummies(
    customer,
    columns=["country"],
    drop_first=True
)

# ============================================================
# DROP UNUSED COLUMNS
# ============================================================

customer.drop(
    columns=[
        "FirstPurchase",
        "LastPurchase"
    ],
    inplace=True
)

# ============================================================
# FEATURES & TARGET
# ============================================================

X = customer.drop(
    columns=[
        "customer_id",
        "Target"
    ]
)

y = customer["Target"]

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

# ============================================================
# FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

joblib.dump(
    scaler,
    "scaler.pkl"
)

print("\nScaler Saved Successfully")

# ============================================================
# SAVE TRAINING DATA (Optional)
# ============================================================

customer.to_csv(
    "customer_training_data.csv",
    index=False
)

print("\nCustomer Training Data Saved")

print("\nData Preparation Completed Successfully!")

Loading dataset...
Dataset Shape : (89139, 10)
Customers     : 500

Creating Training Dataset...
Creating Target Variable...

Target Distribution
Target
0    249
1    203
Name: count, dtype: int64

Training Shape : (361, 26)
Testing Shape  : (91, 26)

Scaler Saved Successfully

Customer Training Data Saved

Data Preparation Completed Successfully!


In [3]:
# ============================================================
# MODEL TRAINING (FAST VERSION)
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import joblib
import pandas as pd

print("\nTraining Models...\n")

# ============================================================
# MODELS
# ============================================================

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=500,
        solver="liblinear",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=50,      # Faster than 200
        max_depth=10,
        n_jobs=-1,            # Use all CPU cores
        random_state=42
    )
}

# ============================================================
# TRAINING LOOP
# ============================================================

results = []

best_model = None
best_name = ""
best_auc = 0

for name, model in models.items():

    print("=" * 60)
    print(f"Training {name}...")

    # Logistic Regression needs scaled data
    if name == "Logistic Regression":

        model.fit(X_train_scaled, y_train)

        prediction = model.predict(X_test_scaled)

        probability = model.predict_proba(
            X_test_scaled
        )[:, 1]

    else:

        model.fit(X_train, y_train)

        prediction = model.predict(X_test)

        probability = model.predict_proba(
            X_test
        )[:, 1]

    # -------------------------------
    # METRICS
    # -------------------------------

    accuracy = accuracy_score(y_test, prediction)

    precision = precision_score(
        y_test,
        prediction,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        prediction,
        zero_division=0
    )

    auc = roc_auc_score(
        y_test,
        probability
    )

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC AUC  : {auc:.4f}")

    results.append({

        "Model": name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "ROC AUC": auc

    })

    if auc > best_auc:

        best_auc = auc

        best_model = model

        best_name = name

        best_prediction = prediction

        best_probability = probability

# ============================================================
# BEST MODEL
# ============================================================

print("\n" + "=" * 60)
print(f"Best Model : {best_name}")
print(f"Best ROC AUC : {best_auc:.4f}")
print("=" * 60)

# ============================================================
# SAVE MODEL
# ============================================================

joblib.dump(
    best_model,
    "return_prediction.pkl"
)

print("\nModel Saved Successfully!")

# ============================================================
# MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)

print("\nModel Comparison\n")

print(results_df)

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report\n")

print(
    classification_report(
        y_test,
        best_prediction
    )
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    best_prediction
)

print("\nConfusion Matrix\n")

print(cm)


Training Models...

Training Logistic Regression...
Accuracy : 0.7033
Precision: 0.6667
Recall   : 0.6829
F1 Score : 0.6747
ROC AUC  : 0.7400
Training Decision Tree...
Accuracy : 0.6593
Precision: 0.6087
Recall   : 0.6829
F1 Score : 0.6437
ROC AUC  : 0.6524
Training Random Forest...
Accuracy : 0.6703
Precision: 0.6341
Recall   : 0.6341
F1 Score : 0.6341
ROC AUC  : 0.7320

Best Model : Logistic Regression
Best ROC AUC : 0.7400

Model Saved Successfully!

Model Comparison

                 Model  Accuracy  Precision    Recall  F1 Score   ROC AUC
0  Logistic Regression  0.703297   0.666667  0.682927  0.674699  0.740000
1        Decision Tree  0.659341   0.608696  0.682927  0.643678  0.652439
2        Random Forest  0.670330   0.634146  0.634146  0.634146  0.731951

Classification Report

              precision    recall  f1-score   support

           0       0.73      0.72      0.73        50
           1       0.67      0.68      0.67        41

    accuracy                           

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# --------------------------------------------------
# PAGE CONFIG
# --------------------------------------------------

st.set_page_config(
    page_title="Customer Return Prediction",
    page_icon="🤖",
    layout="wide"
)

# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------

try:
    model = joblib.load("model/return_prediction.pkl")
    scaler = joblib.load("model/scaler.pkl")
except Exception as e:
    st.error("Model files not found.")
    st.stop()

# --------------------------------------------------
# PAGE TITLE
# --------------------------------------------------

st.title("🤖 Customer Return Prediction")

st.caption(
    "Predict whether a customer is likely to make another purchase."
)

st.divider()

# --------------------------------------------------
# USER INPUT
# --------------------------------------------------

left, right = st.columns(2)

with left:

    total_orders = st.number_input(
        "Total Orders",
        min_value=1,
        value=5
    )

    total_revenue = st.number_input(
        "Total Revenue (£)",
        min_value=0.0,
        value=500.0
    )

    total_quantity = st.number_input(
        "Total Quantity",
        min_value=1,
        value=20
    )

    average_order = st.number_input(
        "Average Order Value",
        min_value=0.0,
        value=100.0
    )

with right:

    recency = st.number_input(
        "Recency (Days)",
        min_value=0,
        value=20
    )

    customer_lifetime = st.number_input(
        "Customer Lifetime (Days)",
        min_value=1,
        value=180
    )

    purchase_frequency = st.number_input(
        "Purchase Frequency",
        min_value=0.0,
        value=0.20,
        format="%.4f"
    )

st.divider()